In [1]:
import weaviate
from weaviate.classes.config import Configure

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Movie",
        vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
            api_endpoint="http://ollama:11434",  # If using Docker you might need: http://host.docker.internal:11434
            model="nomic-embed-text",  # The model to use
        ),
    )

    # Step 1.3: Import three objects
    data_objects = [
        {"title": "The Matrix", "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.", "genre": "Science Fiction"},
        {"title": "Spirited Away", "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.", "genre": "Animation"},
        {"title": "The Lord of the Rings: The Fellowship of the Ring", "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth.", "genre": "Fantasy"},
    ]

    movies = client.collections.use("Movie")
    with movies.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj)

    print(f"Imported & vectorized {len(movies)} objects into the Movie collection")

Imported & vectorized 3 objects into the Movie collection


In [2]:
import weaviate
import json

# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform a semantic search with NearText
    response = movies.query.near_text(
        query="sci-fi",
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "title": "The Matrix",
  "genre": "Science Fiction",
  "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers."
}
{
  "title": "The Lord of the Rings: The Fellowship of the Ring",
  "genre": "Fantasy",
  "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth."
}


In [3]:
import weaviate
from weaviate.classes.generate import GenerativeConfig

# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform RAG with on NearText results
    response = movies.generate.near_text(
        query="sci-fi",
        limit=1,
        grouped_task="Write a tweet with emojis about this movie.",
        generative_provider=GenerativeConfig.ollama(  # Configure the Ollama generative integration
            api_endpoint="http://ollama:11434",  # If using Docker you might need: http://host.docker.internal:11434
            model="llama3.2",  # The model to use
        ),
    )

    print(response.generative.text)  # Inspect the results

"Just watched #TheMatrix 🤖💻 and I'm still reeling from the mind-bending revelations! Neo's journey to uncover the truth about reality is a must-see for sci-fi fans 🤯. Will you take the red pill? 🍡 #TheMatrixLegacy #ScienceFictionClassic"


In [4]:
weaviate.__version__

'4.20.4'

## Weaviate Cloud Experimentation

In [8]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth

load_dotenv()


# Best practice: store your credentials in environment variables
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
)

print(client.is_ready())

True


# Hand-on Weaviate with Python

In [ ]:
import weaviate
import os
from dotenv import load_dotenv

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

client = weaviate.connect_to_local(headers=headers)

/Users/ediwijaya/Code/rag-llm/.venv/lib/python3.13/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/var/folders/4l/j3b6k7xs0rg8p7d0hl3t8wbw0000gp/T/ipykernel_61533/4163605325.py:11: ResourceWarning: unclosed <ssl.SSLSocket fd=90, family=2, type=1, proto=0, laddr=('192.168.1.105', 60654), raddr=('34.160.174.119', 443)>
  client = weaviate.connect_to_local(headers=headers)


## Create Collection

In [12]:
from pyexpat import model
from weaviate.classes.config import Configure, Property, DataType

with weaviate.connect_to_local() as client:
    client.collections.create(
        name="Movies",
        properties=[
            Property(name="title", data_type=DataType.TEXT),
            Property(name="overview", data_type=DataType.TEXT),
            Property(name="vote_average", data_type=DataType.NUMBER),
            Property(name="genre_ids", data_type=DataType.INT_ARRAY),
            Property(name="release_date", data_type=DataType.DATE),
            Property(name="tmdb_id", data_type=DataType.INT),
        ],
        # Define the vectorizer module
        vector_config=Configure.Vectors.text2vec_cohere(model="embed-v4.0"),
        # Define the generative module
        generative_config=Configure.Generative.cohere(model="command-a-03-2025")
    )

In [18]:
with weaviate.connect_to_local() as client:
    collections = client.collections.list_all()
    for name in collections:
        print(name)

Movie
Movies


In [23]:
import os
import json
import requests
from datetime import timezone, datetime
from dotenv import load_dotenv

from tqdm import tqdm
import pandas as pd
import weaviate
from weaviate.util import generate_uuid5

load_dotenv()

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

client = weaviate.connect_to_local(headers=headers)

data_url = "https://raw.githubusercontent.com/weaviate-tutorials/edu-datasets/main/movies_data_1990_2024.json"
respond = requests.get(data_url)
df = pd.DataFrame(respond.json())

movies = client.collections.use("Movies")

with movies.batch.fixed_size(batch_size=200) as batch:
    for i, movie in tqdm(df.iterrows()):
        movie_obj = {
            "title": movie["title"],
            "overview": movie["overview"],
            "vote_average": movie["vote_average"],
            "genre_ids": json.loads(movie["genre_ids"]),
            "release_date": datetime.fromisoformat(movie["release_date"]).replace(tzinfo=timezone.utc),
            "tmdb_id": movie["id"],
        }
        batch.add_object(
            properties=movie_obj,
            uuid=generate_uuid5(movie["id"])
        )
        # batcher automatically sends batches

if len(movies.batch.failed_objects) > 0:
    print(f"Failed to import {len(movies.batch.failed_objects)} objects")

client.close()

680it [00:00, 12139.19it/s]


### Semantic Search

In [ ]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

client = weaviate.connect_to_local(headers=headers)

movies = client.collections.use("Movies")

response = movies.query.near_text(
    query="dystopian future",
    limit=5,
    return_metadata=MetadataQuery(distance=True, certainty=True, score=True)
)

for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)
    print(f"Distance to query: {o.metadata.distance:.3f}")
    print(f"Certainty: {o.metadata.certainty:.3f}, Score: {o.metadata.score:.3f}\n")

client.close()

Gattaca 1997
Distance to query: 0.587
Certainty: 0.706, Score: 0.000

In Time 2011
Distance to query: 0.587
Certainty: 0.706, Score: 0.000

Her 2013
Distance to query: 0.606
Certainty: 0.697, Score: 0.000

I, Robot 2004
Distance to query: 0.613
Certainty: 0.693, Score: 0.000

Looper 2012
Distance to query: 0.617
Certainty: 0.692, Score: 0.000



### BM25 Search

In [32]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    response = movies.query.bm25(
        query="history",
        limit=5,
        return_metadata=MetadataQuery(distance=True, certainty=True, score=True)
    )

    for o in response.objects:
        print(o.properties["title"], o.properties["release_date"].year)
        print(f"BM25 score: {o.metadata.score:.3f}\n")

A Beautiful Mind 2001
BM25 score: 2.723

Legends of the Fall 1994
BM25 score: 2.483

Night at the Museum 2006
BM25 score: 2.412

Hacksaw Ridge 2016
BM25 score: 2.367

The Butterfly Effect 2004
BM25 score: 2.202



### Hybrid search

In [34]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    response = movies.query.hybrid(
        query="history",
        limit=5,
        return_metadata=MetadataQuery(distance=True, certainty=True, score=True)
    )

    for o in response.objects:
        print(o.properties["title"], o.properties["release_date"].year)
        print(f"Hybrid score: {o.metadata.score:.3f}\n")

Legends of the Fall 1994
Hybrid score: 0.936

The Butterfly Effect 2004
Hybrid score: 0.739

American History X 1998
Hybrid score: 0.709

Captain Marvel 2019
Hybrid score: 0.678

A Beautiful Mind 2001
Hybrid score: 0.572



### Filters

In [36]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    response = movies.query.near_text(
        query="distopian future",
        limit=5,
        return_metadata=MetadataQuery(distance=True, certainty=True, score=True),
        filters=Filter.by_property("release_date").greater_than(datetime(2020, 1, 1))
    )

    for o in response.objects:
        print(o.properties["title"], o.properties["release_date"].year)
        print(f"Distance to query: {o.metadata.distance:.3f}\n")

/Users/ediwijaya/Code/rag-llm/.venv/lib/python3.13/site-packages/weaviate/warnings.py:256: UserWarning: Con002: You are using the datetime object 2020-01-01 00:00:00 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warnings.warn(


Dune 2021
Distance to query: 0.688

Jurassic World Dominion 2022
Distance to query: 0.716

The Flash 2023
Distance to query: 0.729

The Adam Project 2022
Distance to query: 0.731

Greenland 2020
Distance to query: 0.737



## LLMs and Weaviate RAG
### 'Single Prompt` Generation

In [48]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    response = movies.generate.near_text(
        query="distopian future",
        limit=5,
        single_prompt="Translate this into French: {title}"
    )

    for o in response.objects:
        print(o.properties["title"])
        print(o.generated)

In Time
"In Time" translates to "En temps" in French. However, if you're referring to the 2011 science fiction film "In Time" directed by Andrew Niccol, the French title is **"Time Out"**.
Her
The translation of "Her" into French depends on the context, as it can be a pronoun or a possessive adjective. Here are the possible translations:

1. **As a pronoun (subject):**  
   - **Elle** (feminine singular)  

2. **As a possessive adjective:**  
   - **Son** (masculine singular, when the noun is masculine)  
   - **Sa** (feminine singular, when the noun is feminine)  
   - **Ses** (plural, for both masculine and feminine nouns)  

Examples:  
- "Her book" → **Son livre** (if the book is masculine) or **Son bouquin** (if using a masculine noun).  
- "Her car" → **Sa voiture**.  
- "Her friends" → **Ses amis** (masculine) or **Ses amies** (feminine).  

Let me know if you need further clarification!
Gattaca
Le titre "Gattaca" reste inchangé en français, car il s'agit d'un nom propre et d'un

In [49]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.query import Filter, MetadataQuery

load_dotenv(override=True)

headers = {
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")
}

with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    response = movies.generate.near_text(
        query="distopian future",
        limit=5,
        grouped_task="What do these movies have in common?"
    )

    for o in response.objects:
        print(o.properties["title"])
        print(o.generated)

In Time

Her

Gattaca

Looper

I, Robot



In [50]:
with weaviate.connect_to_local(headers=headers) as client:
    config = client.collections.get("Movies").config.get()
    print(f"Generative Module: {config.generative_config}")

Generative Module: _GenerativeConfig(generative=<GenerativeSearches.COHERE: 'generative-cohere'>, model={'model': 'command-a-03-2025'})


### Testing using Google Gemini

In [54]:
import os
from dotenv import load_dotenv
import weaviate
from weaviate.classes.generate import GenerativeConfig

load_dotenv(override=True)

# 1. Update headers to use Google API Key
headers = {
    "X-Goog-Api-Key": os.getenv("GOOGLE_API_KEY"),   # For Gemini Generation
    "X-Cohere-Api-Key": os.getenv("COHERE_APIKEY")   # For Cohere Search/Vectorization
}


with weaviate.connect_to_local(headers=headers) as client:
    movies = client.collections.use("Movies")

    # 2. Use Gemini 1.5 for generation
    response = movies.generate.near_text(
        query="sci-fi space exploration",
        limit=3,
        grouped_task="What are the main scientific concepts in these movies?",
        generative_provider=GenerativeConfig.google(
            model="gemini-3-flash-preview"
        )
    )

    print(response.generative.text)


/var/folders/4l/j3b6k7xs0rg8p7d0hl3t8wbw0000gp/T/ipykernel_61533/1203331830.py:23: DeprecationWarning: `google()` is deprecated and will be removed after Q3 '26. Use a service-specific method instead, such as `google_vertex` or `google_gemini`.
  generative_provider=GenerativeConfig.google(


These three films explore a variety of scientific concepts, ranging from theoretical physics and general relativity to astrobiology and engineering. Here are the main scientific concepts featured in each:

### 1. Interstellar (2014)
*Interstellar* is noted for its high level of scientific accuracy, as the production consulted with Nobel Prize-winning physicist Kip Thorne.

*   **General Relativity and Time Dilation:** Because of the intense gravitational pull of the black hole (Gargantua), time passes much slower for the astronauts on nearby planets than for people on Earth.
*   **Wormholes:** The film uses the concept of an Einstein-Rosen Bridge as a shortcut through spacetime, allowing travel between two distant points in the universe.
*   **Black Holes and Event Horizons:** The film provides a scientifically grounded visualization of a rotating black hole, including the "accretion disk" of glowing gas and the "event horizon" from which light cannot escape.
*   **The Tesseract and Hi